Idea: see if we can extend Lowie's search with other information: 
- people who are in Avation network and NOT in Legal network, but are linked to a claim, as actor. 
    - which kinds of claims are there? 
    - which features have these claims? 
    - which features have actor relations? (what does it mean "actor")? 
- people who are in Avation network and NOT in Legal netowrk, but are linked to a claim, as 
target. 
    - which kinds of claims are there? 
    - which features have these claims? 
    - which features have actor relations? (what does it mean "actor")? 

with claim = one extracted statement or assertion. A claim roughly corresponds to one subject-predicate-object style extraction. (total claims over whole graph = 106.012)


Other element: 
- Ghislaine Maxwell was sentenced to prison for 20 years for facilitation of sexual abuse of 
minors, linked to the Epstein files. 
- She appears in the aviation network, but not in the legal network, how come? 

In [2]:
# imports
from gqlalchemy import Memgraph
import pandas as pd

In [3]:
memgraph = Memgraph("127.0.0.1", 7687)

0. Lowie's Aviation network (Aviation_actions) and Legal network (legal_actions)

In [4]:
AVIATION_ACTIONS = [
    'traveled with', 'accompanied', 'flew', 'flew with', 'flew on',
    'flew to', 'traveled on aircraft with', 'traveled on private jet of',
    'flew as passenger on', 'traveled on'
]

LEGAL_ACTIONS = [
    'testified before', 'investigated', 'accused', 'arrested', 'charged',
    'prosecuted', 'negotiated non-prosecution agreement with',
    'signed non-prosecution agreement with', 'was charged with', 'was arrested by'
]

In [22]:
# creation of functions for easy running of Cypher

# run and create df
def run_df(query):
    return pd.DataFrame(memgraph.execute_and_fetch(query))


# run and just print
def run_print(query, limit=None):
    results = list(memgraph.execute_and_fetch(query))

    if limit:
        results = results[:limit]

    for row in results:
        print(row)

1.1 Target and Actor (dges) have NO features: 
Results => no features <=> we need to look at claims. 

In [24]:
run_print("""
MATCH (:Claim)-[r:ACTOR|TARGET]->(:Entity)
UNWIND keys(r) AS feature
RETURN
  type(r) AS relation_type,
  feature,
  count(*) AS number_of_relationships
ORDER BY relation_type, number_of_relationships DESC;
""")

1.2 a. Available Node Features for Claims (node):
Results: Claims have 8 to 10 features: (amount of Claims with this feature is displayed, overal total Claims = 106.012)

In [26]:
run_df("""
MATCH (c:Claim)
UNWIND keys(c) AS property
RETURN property, count(*) AS number_of_claims
ORDER BY number_of_claims DESC;
""")

,property,number_of_claims
0,top_cluster_ids,106012
1,explicit_topic,106012
2,triple_tags,106012
3,sequence_order,106012
4,action,106012
5,claim_id,106012
6,created_at,106012
7,implicit_topic,106011
8,timestamp,61006
9,actor_likely_type,11096


In [28]:
# Example of features Claims: 
run_df("""
MATCH (c:Claim)
WITH c, size(keys(c)) AS n_props
ORDER BY n_props DESC
LIMIT 10
UNWIND keys(c) AS feature
RETURN
  feature,
  c[feature] AS content;
""")


,feature,content
0,created_at,2025-11-14 06:52:20
1,claim_id,HOUSE_OVERSIGHT_019111_part2:0
2,action,interviewed
3,sequence_order,0
4,explicit_topic,interview regarding recruitment of minors
...,...,...
95,implicit_topic,compliance with litigation demands
96,top_cluster_ids,[]
97,triple_tags,"[document_production, discovery_response, fami..."
98,timestamp,2015-03-24


1.3 Entities themselves: which features do they have? 
(entity = canonical real-world entities such as people, organizations, places, 
companies, aircraft, properties, and sometimes unresolved named things.) (55,922 in total) 

In [29]:
run_df("""
MATCH (e:Entity)
UNWIND keys(e) AS feature
RETURN
  feature,
  count(*) AS frequency
ORDER BY frequency DESC;
""")

,feature,frequency
0,name,55922
1,hop_distance_from_principal,26690
2,created_at,26690
3,entity_type,3095


In [30]:
# all entity types and amount of recurrence
run_df("""
MATCH (e:Entity)
WHERE e.entity_type IS NOT NULL
RETURN
  e.entity_type,
  count(*) AS count
ORDER BY count DESC
LIMIT 75;
""")


,e.entity_type,count
0,victim,296
1,government official,248
2,business associate,183
3,government_official,129
4,staff member,129
...,...,...
70,political_activist,5
71,economist,4
72,protester,4
73,plaintiff,4


In [32]:
# are all entities linked in Aviation network persons?
run_df(f"""
MATCH (e:Entity)-[r:RELATED_TO]-()
WHERE r.action IN {AVIATION_ACTIONS}

RETURN
  e.entity_type,
  count(*) AS count

ORDER BY count DESC;
""")


,e.entity_type,count
0,None,176
1,victim,158
2,government official,107
3,model,29
4,defendant,15
5,employee,10
6,political_operator,10
7,legal counsel,9
8,political operator,9
9,associate,6


In [33]:
# entities in aviation network without entity type:
run_df(f"""
MATCH (e:Entity)-[r:RELATED_TO]-()
WHERE r.action IN {AVIATION_ACTIONS}
  AND e.entity_type IS NULL

RETURN DISTINCT
  e.name AS entity_name
ORDER BY entity_name;
""")

,entity_name
0,Abbie
1,Adriana Mucinska
2,Air Force One
3,Al Gore
4,Allie Hanley
...,...
105,one female (unidentified) (HOUSE_OVERSIGHT_021...
106,private 727 plane
107,private plane
108,sayeret team


1.4 Top 20 entities in Aviation network, who are not appearing in Legal network. 
with in return as well: 
- based on which aviation_actions they were included in Aviation network; 
- top 5 of claim_actions (claim feature)
- top 5 of implicit_topic and explicit_topic (claim feature)

In [38]:
run_df(f"""
MATCH (p:Entity)-[r:RELATED_TO]-()
WHERE r.action IN {AVIATION_ACTIONS}

WITH
  p,
  count(DISTINCT r) AS aviation_degree,
  collect(DISTINCT r.action) AS aviation_actions

OPTIONAL MATCH (p)-[lr:RELATED_TO]-()
WHERE lr.action IN {LEGAL_ACTIONS}

WITH p, aviation_degree, aviation_actions, count(lr) AS legal_links
WHERE legal_links = 0

MATCH (claim:Claim)-[role:ACTOR|TARGET]->(p)

WITH
  p,
  aviation_degree,
  aviation_actions,
  count(DISTINCT claim) AS total_claims,
  count(CASE WHEN type(role) = 'ACTOR' THEN 1 END) AS claims_as_actor,
  count(CASE WHEN type(role) = 'TARGET' THEN 1 END) AS claims_as_target,
  collect(claim.action) AS claim_actions,
  collect(claim.explicit_topic) AS explicit_topics,
  collect(claim.implicit_topic) AS implicit_topics

RETURN
  p.name AS person,
  p.entity_type AS entity_type,
  p.hop_distance_from_principal AS hop_distance_from_principal,
  aviation_degree,
  aviation_actions,
  total_claims,
  claims_as_actor,
  claims_as_target,
  claim_actions[0..5] AS sample_top5_claim_actions,
  explicit_topics[0..5] AS sample_top5_explicit_topics,
  implicit_topics[0..5] AS sample_top5_implicit_topics

ORDER BY aviation_degree DESC
LIMIT 20;
""")

,person,entity_type,hop_distance_from_principal,aviation_degree,aviation_actions,total_claims,claims_as_actor,claims_as_target,sample_top5_claim_actions,sample_top5_explicit_topics,sample_top5_implicit_topics
0,Ghislaine Maxwell,defendant,1.0,15,"[traveled with, flew with, accompanied, flew]",1285,817,468,"[attended event with, attended reception with,...","[social event attendance, social event attenda...","[relationship cultivation with associate, high..."
1,Kevin Spacey,None,1.0,11,"[traveled with, flew, traveled on aircraft with]",34,5,29,"[traveled on aircraft with, testified that was...","[private aircraft travel as passenger, plane p...","[documented association with accused, entertai..."
2,Lawrence Summers,political operator,NaN,9,"[accompanied, traveled on private jet of, flew...",661,334,327,"[had friendship with, wrote letter of commitme...","[personal friendship, Harvard president commit...","[policy influence networking, institutional va..."
3,Jeffrey Epstein's jet,None,NaN,7,[flew on],11,0,11,"[traveled_on, flew on, flew on, flew on, flew on]","[air travel on private jet, air travel via pri...","[maintaining elite network, association with E..."
4,Chris Tucker,None,1.0,6,"[flew, traveled on aircraft with, traveled with]",18,5,13,"[traveled on aircraft with, testified that was...","[private aircraft travel as passenger, plane p...","[documented association with accused, celebrit..."
5,Virginia,victim,1.0,5,"[traveled with, flew]",55,19,36,"[reported surveillance and abuse to, agreed to...","[victim disclosure to law enforcement, witness...","[evidence gathering and victim support, buildi..."
6,President Waheed,None,NaN,5,"[accompanied, traveled on]",23,22,1,"[attended, met with, met with, attended, dined...","[attendance at CMAG meeting, meeting with UNGA...","[regional security coordination, political net..."
7,Jeffrey Epstein's private plane,None,2.0,4,"[flew on, traveled on]",4,0,4,"[traveled on, flew on, flew on, flew on]",[documentation of victim's presence on defenda...,[establishing proximity and opportunity for ab...
8,Adriana Mucinska,None,2.0,4,"[traveled with, flew with]",5,1,4,"[traveled with, flew with, flew with, traveled...","[shared flight with Mucinska, shared flights w...","[association with Epstein staff, association w..."
9,Naomi Campbell,None,1.0,4,"[traveled with, flew with, flew as passenger on]",18,6,12,"[requested exclusive representation from, disc...","[model seeking exclusive agent, discovery of s...","[building professional relationship, career ad..."


1.5 Top 100 Claim actions with top 10 explicit and implicit topics and 

In [40]:
run_df(f"""
MATCH (p:Entity)-[r:RELATED_TO]-()
WHERE r.action IN {AVIATION_ACTIONS}
WITH 
  p, 
  count(DISTINCT r) AS aviation_degree, 
  collect(DISTINCT r.action) AS aviation_actions

OPTIONAL MATCH (p)-[lr:RELATED_TO]-()
WHERE lr.action IN {LEGAL_ACTIONS}

WITH p, aviation_degree, count(lr) AS legal_links
WHERE legal_links = 0
WITH p, aviation_degree
ORDER BY aviation_degree DESC
LIMIT 20

WITH collect(p) AS top_people

MATCH (claim:Claim)-[:ACTOR|TARGET]->(p)
WHERE p IN top_people AND claim.action IS NOT NULL
WITH top_people, claim.action AS action, count(DISTINCT claim) AS action_count
ORDER BY action_count DESC
LIMIT 100

MATCH (claim:Claim)-[:ACTOR|TARGET]->(p)
WHERE p IN top_people
  AND claim.action = action
  AND claim.explicit_topic IS NOT NULL
WITH top_people, action, action_count, claim.explicit_topic AS explicit_topic, count(DISTINCT claim) AS explicit_count
ORDER BY action, explicit_count DESC
WITH top_people, action, action_count, collect([explicit_topic, explicit_count])[0..10] AS top10_explicit_topics

MATCH (claim:Claim)-[:ACTOR|TARGET]->(p)
WHERE p IN top_people
  AND claim.action = action
  AND claim.implicit_topic IS NOT NULL
WITH action, action_count, top10_explicit_topics, claim.implicit_topic AS implicit_topic, count(DISTINCT claim) AS implicit_count
ORDER BY action, implicit_count DESC

WITH
  action,
  action_count,
  top10_explicit_topics,
  collect([implicit_topic, implicit_count])[0..10] AS top10_implicit_topics

UNWIND top10_explicit_topics AS explicit_item
UNWIND top10_implicit_topics AS implicit_item

RETURN
  action,
  action_count,

  explicit_item[0] AS explicit_topic,
  explicit_item[1] AS explicit_topic_count,

  implicit_item[0] AS implicit_topic,
  implicit_item[1] AS implicit_topic_count

ORDER BY action_count DESC,
         explicit_topic_count DESC,
         implicit_topic_count DESC;
""")

,action,action_count,explicit_topic,explicit_topic_count,implicit_topic,implicit_topic_count
0,sent email to,304,assessment of Trump's legal and political vuln...,2,relationship maintenance through social engage...,2
1,sent email to,304,inquiry about Trump's current political status,2,relationship maintenance through social engage...,2
2,sent email to,304,inquiry about Trump's current political status,2,personal relationship maintenance and social e...,2
3,sent email to,304,assessment of Trump's legal and political vuln...,2,personal relationship maintenance and social e...,2
4,sent email to,304,introduction to United Nations official,2,personal relationship maintenance and social e...,2
...,...,...,...,...,...,...
2829,arrived at,3,arrival in Dubai,1,supporting Snowden,1
2830,paid,3,payment to attorney,1,influence over legal representation,1
2831,paid,3,payment to attorney,1,facilitating sexual exploitation,1
2832,paid,3,payment to attorney,1,compensation for sexual activities,1
